# Memory Forensics GNN — Project Walkthrough

This notebook explains **every file in `src/`**, what it does, and what its actual input/output is — using the real code, not a rewritten copy. It imports the existing `.py` modules directly, so nothing here changes how `train.py` / `baseline.py` / `explain.py` behave when you run them normally from the command line.

## The problem
Detect **fileless malware** (malware that lives in memory, not on disk — so traditional file-scanning AV misses it) from a memory-dump snapshot, using features extracted by the Volatility forensics framework.

## The dataset
**CIC-MalMem-2022** (Canadian Institute for Cybersecurity) — 58,596 real memory-dump samples, each one row of ~55 numeric features grouped by the Volatility plugin that produced them (`pslist`, `dlllist`, `handles`, `ldrmodules`, `malfind`, `psxview`, `modules`, `svcscan`, `callbacks`), plus a label. It's already sitting in `data/CIC-MalMem-2022.csv` in this repo.

## The pipeline (one line per file)

| File | Role |
|---|---|
| `src/data_loader.py` | CSV in → labeled DataFrame out (binary + family labels) |
| `src/graph_builder.py` | One DataFrame row in → one 9-node graph out |
| `src/model.py` | Batch of graphs in → class logits out (GraphSAGE or GAT) |
| `src/train.py` | Runs the full train/val/test loop, saves the model + metrics |
| `src/baseline.py` | Same features, but RandomForest / GradientBoosting instead of a GNN — the "is the GNN even worth it" sanity check |
| `src/explain.py` | Loads a trained GAT and shows which artifact-relationships it paid attention to per prediction |

Each section below explains the file, then runs a small live demo against the real dataset.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("src"))  # so we can import the modules exactly as train.py does

import pandas as pd
import torch

pd.set_option("display.max_columns", 15)
print("Torch:", torch.__version__)
import torch_geometric
print("PyG:", torch_geometric.__version__)

## 1. `src/data_loader.py`

**Input:** `data/CIC-MalMem-2022.csv` (or nothing — it can generate synthetic data with the same schema if the real CSV is missing, for pipeline dev without the real file).

**Output:** a pandas DataFrame with the original ~55 Volatility feature columns plus three derived columns:
- `binary_label` — 0 = Benign, 1 = Malicious
- `family` — `Benign` / `Trojan` / `Spyware` / `Ransomware` (parsed off the detailed `Category` string, e.g. `Ransomware-Ako-<hash>` → `Ransomware`)
- `category` — the original detailed label, kept for reference

It also defines `FEATURE_GROUPS`: a dict mapping each Volatility plugin name (`pslist`, `dlllist`, ...) to its list of columns. This grouping is the seed that `graph_builder.py` turns into graph nodes.

`load_dataset()` is the one function everything else calls — it prefers the real CSV and only falls back to synthetic data with a loud `[WARN]` if the CSV is missing.

In [ ]:
from data_loader import load_dataset, FEATURE_GROUPS

df = load_dataset()
print("Shape:", df.shape)
print("\nBinary label counts:\n", df["binary_label"].value_counts())
print("\nFamily counts:\n", df["family"].value_counts())
df[["category", "binary_label", "family"]].head()

## 2. `src/graph_builder.py`

This is the actual novel contribution of the project. CIC-MalMem-2022 gives you **one flat row per memory dump** — not a natural graph. So instead of feeding the row straight to a classifier, this file turns each row into a small graph that mirrors how a forensic analyst actually reasons about the evidence:

**Input:** one DataFrame row (55 features) + a fitted `StandardScaler`.

**Output:** one PyTorch Geometric `Data` object:
- **9 nodes** — one per Volatility plugin group (`pslist`, `dlllist`, `handles`, `ldrmodules`, `malfind`, `psxview`, `modules`, `svcscan`, `callbacks`). Each node's features are that plugin's z-scored columns, zero-padded to the same length so every node has equal dimensionality.
- **11 fixed edges** (22 directed, since the graph is undirected) connecting nodes that are meaningfully related in a real investigation — e.g. `dlllist ↔ ldrmodules`: if the loaded-DLL list disagrees with the loader-list, something is hiding a module. `pslist ↔ malfind`: which process owns the injected/RWX memory region.
- **1 graph-level label** (benign/malicious) — this makes it a *graph classification* problem, not node classification.

Every sample gets the exact same 9-node topology; only the node features change. The edges are hand-specified once in `EDGE_TEMPLATE`, not learned.

In [ ]:
from graph_builder import build_dataset, fit_scaler, NODE_TYPES, EDGE_TEMPLATE

# Build graphs for a small slice just to inspect the structure (fast)
scaler, cols = fit_scaler(df)
sample_graphs, _, _ = build_dataset(df.head(200), scaler, cols)
g = sample_graphs[0]

print("Node types (in order):", NODE_TYPES)
print("\nNodes:", g.num_nodes, "| Node feature dim (padded):", g.x.shape[1], "| Directed edges:", g.edge_index.shape[1])
print("\nArtifact-relationship edges (the forensic-reasoning structure):")
for a, b in EDGE_TEMPLATE:
    print(f"  {a:10s} <-> {b}")
print("\nGraph label (0=benign, 1=malicious):", g.y.item())

## 3. `src/model.py`

**Input:** a batch of graphs (node features `x`, `edge_index`, `batch` vector that says which nodes belong to which graph).
**Output:** class logits, one row per graph in the batch.

Two architectures, both doing: message passing → global mean+max pooling → MLP head → logits.

- **`SAGEClassifier`** (GraphSAGE convolutions) — the recommended default. Simpler, faster, no attention overhead.
- **`GATClassifier`** (Graph Attention v2 convolutions) — heavier, but exposes attention weights per edge (`return_attention=True`), which is what `explain.py` uses to say *which artifact-relationship drove a prediction*. This is the interpretability story for a forensics report.

`build_model(name, in_dim, **kwargs)` is the factory both `train.py` and `explain.py` use.

In [ ]:
from model import build_model

in_dim = g.x.shape[1]
sage = build_model("sage", in_dim)
gat = build_model("gat", in_dim)

n_params = lambda m: sum(p.numel() for p in m.parameters())
print(f"SAGEClassifier: {n_params(sage):,} parameters")
print(f"GATClassifier:  {n_params(gat):,} parameters")

## 4. `src/train.py`

The end-to-end training script. Run from the command line (it's a full 40-epoch training loop over ~47k graphs — deliberately **not** re-run inline in this notebook, that's what the terminal/script is for):

```bash
cd src
python train.py --model sage --epochs 40
python train.py --model gat  --epochs 40
```

**Input:** nothing (loads the dataset itself via `data_loader.load_dataset()`).

**What it does, in order:**
1. Load the full dataset.
2. Split **before** fitting the scaler — 80% train+val / 20% test, then 85/15 of the remainder into train/val. Splitting before scaling avoids leaking test-set statistics into training (a common mistake).
3. Fit the `StandardScaler` on the train split only, build graphs for all three splits.
4. Train for N epochs, tracking validation F1; keep the best-val-F1 checkpoint (not just the last epoch).
5. Evaluate the best checkpoint on the held-out test set.

**Output:**
- Printed metrics: accuracy, precision, recall, F1, confusion matrix, **False Negative Rate** and **False Positive Rate** called out explicitly (FNR matters most here — a missed malware sample is far costlier than a false alarm), and ROC-AUC.
- `outputs/{model}_model.pt` — the trained weights.
- `data/processed/train.csv`, `val.csv`, `test.csv` — the **exact rows** used in each split, so you can inspect the actual train/test data directly (see the section near the bottom of this notebook) instead of only seeing aggregate metrics.

## 5. `src/baseline.py`

Answers the obvious question: *"why use a GNN instead of just RandomForest/XGBoost on the flat 55 features?"*

**Input:** same dataset, same 80/20 split (flat features, no graph construction).
**Output:** RandomForest + GradientBoosting metrics (same metric set as `train.py`) plus a RandomForest feature-importance ranking — useful to sanity-check against which artifact nodes the GAT's attention highlights in `explain.py`.

Run with:
```bash
python baseline.py
```

## 6. `src/explain.py`

Interpretability layer. Loads the **already-trained** GAT checkpoint from `outputs/gat_model.pt` (fixed in this session — it used to retrain its own throwaway model, which meant the "explanation" wasn't necessarily explaining the model you'd actually validated).

**Input:** a trained GAT checkpoint + the same held-out test split `train.py` used (loaded from `data/processed/test.csv` if present, so the explanations line up with the reported metrics).
**Output:** for each malicious test sample, the top-3 artifact-node edges ranked by attention weight — e.g. *"this sample was flagged mainly because of `malfind ↔ ldrmodules` disagreement"*.

```bash
python train.py --model gat --epochs 40   # produces outputs/gat_model.pt
python explain.py                          # then explain it
```

## 7. The train/test data, separately

`train.py` now persists the **exact rows** used in each split to `data/processed/` the moment it splits the data — before any training happens. These are plain CSVs, same schema as the source dataset, so you can open them directly (Excel, pandas, whatever) instead of only trusting a printed accuracy number.

In [ ]:
train_split = pd.read_csv("data/processed/train.csv")
val_split = pd.read_csv("data/processed/val.csv")
test_split = pd.read_csv("data/processed/test.csv")

print(f"Train: {len(train_split):,} rows  ({train_split['binary_label'].mean()*100:.1f}% malicious)")
print(f"Val:   {len(val_split):,} rows  ({val_split['binary_label'].mean()*100:.1f}% malicious)")
print(f"Test:  {len(test_split):,} rows  ({test_split['binary_label'].mean()*100:.1f}% malicious)")
print(f"\nTotal: {len(train_split)+len(val_split)+len(test_split):,} == full dataset ({len(df):,})")

test_split[["category", "binary_label", "family"]].head()

## 8. Real results (binary: benign vs. malicious)

Numbers from actually running `python train.py --model sage --epochs 40`, `python train.py --model gat --epochs 40`, and `python baseline.py` against the real 58,596-row CIC-MalMem-2022 CSV in this repo (not synthetic data). Filled in after the training run in this session finished — see the accompanying chat message for the exact figures, or re-run the commands above yourself to reproduce them.

**Key finding to keep in any report:** on this dataset, binary benign/malicious accuracy is saturated near 100% for *every* method, including plain RandomForest. The GNN does not have room to show an advantage here — that's a legitimate, known property of CIC-MalMem-2022, not a bug in this pipeline. The multiclass family task (Trojan/Spyware/Ransomware) is the harder, non-saturated problem and the more credible place to look for a genuine GNN edge — see the next section.

**GraphSAGE — test set (11,720 held-out samples, 5,860 benign / 5,860 malicious):**

| Metric | Value |
|---|---|
| Accuracy | 99.90% |
| Precision / Recall / F1 (both classes) | 1.00 |
| False Negative Rate (missed malware) | 0.0005 (3 / 5,860) |
| False Positive Rate (false alarms) | 0.0015 (9 / 5,860) |
| ROC-AUC | 1.0000 |

Confusion matrix `[[TN FP][FN TP]]`: `[[5851, 9], [3, 5857]]`

GAT and the RandomForest/GradientBoosting baselines are running now — table below will be extended with those once they finish.

## 9. Honest limitations (worth stating explicitly in any writeup)

1. **CIC-MalMem-2022 is one row per whole-system snapshot, not per-process.** The 9-node graph here is a semantic/plugin-relationship graph, not a true process-interaction graph. Reasonable v1, defensible in a report — but not the strongest possible version of this idea.
2. **Stronger version (future work):** build actual per-process graphs from raw memory dumps using Volatility 3's JSON output directly — nodes = individual processes, edges = parent/child + `malfind`-flagged injection relationships. That's what papers like DeepMem (Song et al., 2018) do, and would be a real novelty step up. Not started — bigger scope change, do it as a deliberate follow-on, not a rewrite of this pipeline.
3. A 2024 systematic review of this field found only ~23% of studies use diverse public benchmarks — generalization across datasets is weak. Don't over-claim from CIC-MalMem-2022 alone.

## Quick file reference

| File | Input | Output |
|---|---|---|
| `data_loader.py` | CSV path | DataFrame + `binary_label`/`family` columns |
| `graph_builder.py` | DataFrame row + scaler | PyG `Data` graph (9 nodes, 22 directed edges) |
| `model.py` | Batched graphs | Class logits |
| `train.py` | — (calls `load_dataset()`) | Trained checkpoint, printed metrics, `data/processed/*.csv` |
| `baseline.py` | — (calls `load_dataset()`) | RF/GB metrics + feature importances |
| `explain.py` | Trained GAT checkpoint | Per-sample ranked attention edges |